# Music Annotation Project V8.2F
## Full-Development `p_print` Provider Freeze

**Runtime: GPU.** Select a T4 GPU and run all.

This notebook refits the exact V8.2 33x33 `TopologyCNN` once on the complete stable development population, fits one isotonic calibrator from the complete source-group OOF raw PRINT scores, packages the observable-only runtime provider, and requires exact reload replay. It does not change thresholds, use independent data, or claim production authorisation.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow torch torchvision scikit-learn joblib

In [ ]:
import os
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'; os.environ['PYTHONHASHSEED']='42'
from pathlib import Path
from datetime import datetime,timezone
import json,hashlib,random,shutil,sys,platform,gc,zipfile
import cv2,torch,joblib,sklearn,numpy as np,pandas as pd
from torch import nn
from torch.utils.data import Dataset,DataLoader
from sklearn.isotonic import IsotonicRegression
ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject'); V82_ROOT=ROOT/'observed_print_classifier_v8_2'; OUTPUT_ROOT=ROOT/'observed_print_full_development_freeze_v8_2f'; SOURCE_V82_RUN_OVERRIDE=None
SEED=42; PATCH=33; HALF=16; BATCH=512; EPOCHS=12; LR=1e-3; WEIGHT_DECAY=1e-4
CHANNELS=['obs','norm','mask','depth','horizontal','vertical','compact','preserve','remove','ambiguous','edge']
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE!='cuda': raise RuntimeError('Select a GPU runtime.')
torch.use_deterministic_algorithms(True); torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
OUT=OUTPUT_ROOT/(datetime.now().strftime('%Y%m%d_%H%M%S')+'_v8_2f_full_development_p_print_freeze'); BUNDLE=OUT/'p_print_provider'; REPLAY=BUNDLE/'replay'; TRAIN=BUNDLE/'training_record'; AUDIT=OUT/'audit'; UPLOAD=OUT/'upload'
for p in [OUT,BUNDLE,REPLAY,TRAIN,AUDIT,UPLOAD]: p.mkdir(parents=True,exist_ok=False)
def require(x,m):
 if not bool(x): raise RuntimeError(m)
def sha(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for b in iter(lambda:f.read(1048576),b''): h.update(b)
 return h.hexdigest()
def oh(x): return hashlib.sha256(json.dumps(x,sort_keys=True,default=str,separators=(',',':')).encode()).hexdigest()
def writej(p,x): p.write_text(json.dumps(x,indent=2,sort_keys=True,default=str))
def seed(): random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
def imap(folder):
 ex={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
 return {p.stem:p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in ex}
print('GPU:',torch.cuda.get_device_name(0)); print('Output:',OUT)

## 1. Resolve and verify one complete V8.2 development run

In [ ]:
REQ=['experiment_identity.json','inputs/channel_schema.json','tables/stable_sample_manifest.csv','tables/stable_raw_oof.npy','audit/model_inventory.csv','audit/calibrator_inventory.csv']
def runs(root): return sorted([p for p in root.iterdir() if p.is_dir() and all((p/r).exists() for r in REQ)],key=lambda p:p.stat().st_mtime) if root.exists() else []
SOURCE=Path(SOURCE_V82_RUN_OVERRIDE) if SOURCE_V82_RUN_OVERRIDE else (runs(V82_ROOT)[-1] if runs(V82_ROOT) else None)
require(SOURCE and all((SOURCE/r).exists() for r in REQ),'No complete V8.2 run found.')
identity=json.loads((SOURCE/'experiment_identity.json').read_text()); schema=json.loads((SOURCE/'inputs/channel_schema.json').read_text())
require(identity.get('version')=='8.2-development-whole-page','Wrong V8.2 version.'); require(identity.get('independent_data_used') is False,'Independent data source prohibited.')
require(identity.get('seed')==SEED and identity.get('patch')==PATCH and identity.get('channels')==CHANNELS,'Model/schema contract mismatch.')
require(identity.get('batch')==BATCH and identity.get('epochs')==EPOCHS and float(identity.get('lr'))==LR and float(identity.get('weight_decay'))==WEIGHT_DECAY,'Training parameter mismatch.')
require(schema.get('channels')==CHANNELS and schema.get('patch')==PATCH and schema.get('padding')=='reflect','Channel schema mismatch.')
print('Source:',SOURCE)

## 2. Rebuild exact observable tensors for the frozen stable population

In [ ]:
stable=pd.read_csv(SOURCE/'tables/stable_sample_manifest.csv').sort_values(['source_group','page','y','x']).reset_index(drop=True)
for x in ['y','x','truth']: stable[x]=pd.to_numeric(stable[x],errors='raise').astype(int)
require({'page','source_group','y','x','truth'}.issubset(stable.columns),'Stable manifest fields missing.'); require(set(stable.truth.unique())=={0,1},'Both binary classes required.'); require(not stable.duplicated(['page','y','x']).any(),'Duplicate stable coordinates.')
raw_oof=np.load(SOURCE/'tables/stable_raw_oof.npy').astype(np.float64); require(len(raw_oof)==len(stable) and np.isfinite(raw_oof).all(),'OOF score mismatch.')
V75=Path(identity['v7_5_reference']); status=pd.read_csv(V75/'tables/cache_status.csv'); cache={str(r.page):Path(r.cache) for _,r in status.iterrows()}; observed=imap(ROOT/'aligned'); pages=sorted(stable.page.astype(str).unique())
runtime={}; arrs=['mask','g80_preserve','g80_remove','g80_ambiguous','g80_horizontal','g80_vertical','g80_compact']
for page in pages:
 require(page in cache and page in observed,f'{page}: source arrays missing.')
 with np.load(cache[page],allow_pickle=False) as z:
  require(all(k in z.files for k in arrs),f'{page}: cache fields missing.'); a={k:z[k].copy() for k in arrs}
 obs=cv2.imread(str(observed[page]),0); require(obs is not None and all(v.shape==obs.shape for v in a.values()),f'{page}: shape mismatch.')
 mask=a['mask']>0; vis=(~mask)&(obs>=200); num=cv2.GaussianBlur(obs.astype(np.float32)*vis,(0,0),45.0); den=cv2.GaussianBlur(vis.astype(np.float32),(0,0),45.0); paper=np.clip(np.where(den>.02,num/np.maximum(den,1e-4),cv2.GaussianBlur(obs,(0,0),45.0)),0,255).astype(np.uint8)
 runtime[page]={'obs':obs,'norm':cv2.divide(obs,np.maximum(paper,1),scale=255),'mask':mask,'depth':cv2.distanceTransform(mask.astype(np.uint8),cv2.DIST_L2,5),'horizontal':a['g80_horizontal']>0,'vertical':a['g80_vertical']>0,'compact':a['g80_compact']>0,'preserve':a['g80_preserve']>0,'remove':a['g80_remove']>0,'ambiguous':a['g80_ambiguous']>0,'edge':cv2.Canny(obs,70,170)>0}
padded={p:{k:np.pad((v.astype(np.float32)/255.0 if k in ['obs','norm'] else v.astype(np.float32)),HALF,mode='reflect') for k,v in z.items()} for p,z in runtime.items()}
plist=pages; pindex={p:i for i,p in enumerate(plist)}; pids=stable.page.map(pindex).to_numpy(np.int16); ys=stable.y.to_numpy(np.int32); xs=stable.x.to_numpy(np.int32); truth=stable.truth.to_numpy(np.int8)
def patch(pid,y,x):
 p=plist[int(pid)]; return np.stack([padded[p][k][y:y+PATCH,x:x+PATCH] for k in CHANNELS]).astype(np.float32)
shutil.copy2(SOURCE/'inputs/channel_schema.json',BUNDLE/'channel_schema.json')
writej(TRAIN/'development_population.json',{'rows':len(stable),'pages':stable.page.nunique(),'source_groups':stable.source_group.nunique(),'annotation':int((truth==0).sum()),'print':int((truth==1).sum()),'stable_manifest_sha256':sha(SOURCE/'tables/stable_sample_manifest.csv'),'raw_oof_sha256':sha(SOURCE/'tables/stable_raw_oof.npy')})
print('Rows:',len(stable),'pages:',stable.page.nunique(),'groups:',stable.source_group.nunique())

## 3. Exact CNN, full-development fit and OOF-evidence calibration

In [ ]:
class DS(Dataset):
 def __init__(self,idx): self.idx=np.asarray(idx)
 def __len__(self): return len(self.idx)
 def __getitem__(self,i):
  j=int(self.idx[i]); return torch.from_numpy(patch(pids[j],ys[j],xs[j])),torch.tensor(int(truth[j]),dtype=torch.long)
class TopologyCNN(nn.Module):
 def __init__(self,nc=len(CHANNELS)):
  super().__init__(); self.net=nn.Sequential(nn.Conv2d(nc,24,3,padding=1),nn.ReLU(),nn.BatchNorm2d(24),nn.MaxPool2d(2),nn.Conv2d(24,48,3,padding=1),nn.ReLU(),nn.BatchNorm2d(48),nn.MaxPool2d(2),nn.Conv2d(48,64,3,padding=1),nn.ReLU(),nn.AdaptiveAvgPool2d(1)); self.head=nn.Linear(64,2)
 def forward(self,x): return self.head(self.net(x).flatten(1))
def wi(w): random.seed(SEED+w); np.random.seed(SEED+w)
def loader(idx,shuffle):
 g=torch.Generator(); g.manual_seed(SEED); return DataLoader(DS(idx),batch_size=BATCH,shuffle=shuffle,num_workers=2,pin_memory=True,generator=g,worker_init_fn=wi,persistent_workers=False)
def predict(model,idx):
 out=[]; model.eval()
 with torch.no_grad():
  for xb,_ in loader(idx,False): out.extend(torch.softmax(model(xb.to(DEVICE,non_blocking=True)),1)[:,1].cpu().numpy())
 return np.asarray(out,dtype=np.float64)
seed(); model=TopologyCNN().to(DEVICE); opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY); counts=np.bincount(truth,minlength=2); weights=torch.tensor(counts.sum()/np.maximum(counts,1),dtype=torch.float32,device=DEVICE); lossfn=nn.CrossEntropyLoss(weight=weights); idx=np.arange(len(stable)); hist=[]
for ep in range(EPOCHS):
 model.train(); total=0.0
 for xb,yb in loader(idx,True):
  xb=xb.to(DEVICE,non_blocking=True); yb=yb.to(DEVICE,non_blocking=True); opt.zero_grad(set_to_none=True); loss=lossfn(model(xb),yb); loss.backward(); opt.step(); total+=loss.detach().item()*len(yb)
 hist.append({'epoch':ep+1,'mean_loss':total/len(idx)}); print(ep+1,hist[-1]['mean_loss'])
model_path=BUNDLE/'p_print_model.pt'; torch.save(model.state_dict(),model_path); pd.DataFrame(hist).to_csv(TRAIN/'training_history.csv',index=False)
cal=IsotonicRegression(out_of_bounds='clip').fit(raw_oof,truth); cal_path=BUNDLE/'p_print_calibrator.joblib'; joblib.dump(cal,cal_path)
writej(TRAIN/'exact_model_constructor.json',{'class':'TopologyCNN','channels':CHANNELS,'patch':PATCH,'architecture':'Conv11-24-BN-pool, Conv24-48-BN-pool, Conv48-64, adaptive average pool, Linear64-2'})
writej(TRAIN/'model_fitting_record.json',{'rows':len(stable),'pages':stable.page.nunique(),'source_groups':stable.source_group.nunique(),'seed':SEED,'batch':BATCH,'epochs':EPOCHS,'lr':LR,'weight_decay':WEIGHT_DECAY,'class_counts':counts.tolist(),'class_weights':weights.detach().cpu().numpy().tolist(),'model_sha256':sha(model_path),'role':'post-development full-permitted-data deployment refit'})
writej(TRAIN/'calibrator_fitting_record.json',{'class':'IsotonicRegression','out_of_bounds':'clip','fit_rows':len(stable),'fit_input':'complete V8.2 source-group OOF raw PRINT scores','x_thresholds':cal.X_thresholds_.tolist(),'y_thresholds':cal.y_thresholds_.tolist(),'calibrator_sha256':sha(cal_path)})

## 4. Freeze provider source and exact reload replay

In [ ]:
policy={'provider_name':'V8.2F_FULL_DEVELOPMENT_P_PRINT','output':'calibrated_p_print','ordered_channels':CHANNELS,'patch':PATCH,'padding':'reflect','normalisation':{'obs':'divide_255','norm':'divide_255','other':'native_float32'},'eligibility':{'inside_authoritative_mask':True,'observed_below':200,'not_v78_preserved':True},'model_role':'full-development deployment refit','calibration_role':'complete OOF evidence','clean_truth_required_at_inference':False,'decision_thresholds_in_provider':False}
policy['provider_id']=oh(policy); writej(BUNDLE/'provider_policy.json',policy)
provider_lines=[
 'import cv2, numpy as np, torch','from torch import nn',f'CHANNELS={CHANNELS!r}','PATCH=33','HALF=16',
 'class TopologyCNN(nn.Module):',
 ' def __init__(self,nc=len(CHANNELS)):',
 "  super().__init__(); self.net=nn.Sequential(nn.Conv2d(nc,24,3,padding=1),nn.ReLU(),nn.BatchNorm2d(24),nn.MaxPool2d(2),nn.Conv2d(24,48,3,padding=1),nn.ReLU(),nn.BatchNorm2d(48),nn.MaxPool2d(2),nn.Conv2d(48,64,3,padding=1),nn.ReLU(),nn.AdaptiveAvgPool2d(1)); self.head=nn.Linear(64,2)",
 ' def forward(self,x): return self.head(self.net(x).flatten(1))',
 'def background_surface(obs,mask,sigma=45.0):',
 ' vis=(~mask)&(obs>=200); num=cv2.GaussianBlur(obs.astype(np.float32)*vis,(0,0),sigma); den=cv2.GaussianBlur(vis.astype(np.float32),(0,0),sigma); return np.clip(np.where(den>.02,num/np.maximum(den,1e-4),cv2.GaussianBlur(obs,(0,0),sigma)),0,255).astype(np.uint8)',
 'def build_channels(obs,mask,routes):',
 " paper=background_surface(obs,mask); arrays={'obs':obs,'norm':cv2.divide(obs,np.maximum(paper,1),scale=255),'mask':mask,'depth':cv2.distanceTransform(mask.astype(np.uint8),cv2.DIST_L2,5),'horizontal':routes['horizontal'],'vertical':routes['vertical'],'compact':routes['compact'],'preserve':routes['preserve'],'remove':routes['remove'],'ambiguous':routes['ambiguous'],'edge':cv2.Canny(obs,70,170)>0}",
 " return {k:np.pad((v.astype(np.float32)/255.0 if k in ('obs','norm') else v.astype(np.float32)),HALF,mode='reflect') for k,v in arrays.items()}",
 'def extract(padded,ys,xs): return np.stack([np.stack([padded[k][int(y):int(y)+PATCH,int(x):int(x)+PATCH] for k in CHANNELS]) for y,x in zip(ys,xs)]).astype(np.float32)',
 'def score(model,calibrator,padded,ys,xs,device,batch=512):',
 ' model.eval(); out=[]',
 ' with torch.no_grad():',
 '  for s in range(0,len(ys),batch): out.extend(torch.softmax(model(torch.from_numpy(extract(padded,ys[s:s+batch],xs[s:s+batch])).to(device)),1)[:,1].cpu().numpy())',
 ' raw=np.asarray(out,dtype=np.float64); return raw,np.asarray(calibrator.predict(raw),dtype=np.float64)']
(BUNDLE/'p_print_provider.py').write_text(chr(10).join(provider_lines)+chr(10))
positions=np.asarray(sorted(set([0,len(stable)//4,len(stable)//2,3*len(stable)//4,len(stable)-1]))); expected_raw=predict(model,positions); expected_p=cal.predict(expected_raw).astype(np.float64)
fixture=stable.iloc[positions].copy(); fixture['expected_raw_print']=expected_raw; fixture['expected_p_print']=expected_p; fixture.to_csv(REPLAY/'replay_fixture.csv',index=False)
loaded=TopologyCNN().to(DEVICE); loaded.load_state_dict(torch.load(model_path,map_location=DEVICE),strict=True); loaded_cal=joblib.load(cal_path); actual_raw=predict(loaded,positions); actual_p=loaded_cal.predict(actual_raw).astype(np.float64)
checks={'raw_exact':np.array_equal(expected_raw,actual_raw),'calibrated_exact':np.array_equal(expected_p,actual_p),'finite':np.isfinite(actual_p).all(),'unit_interval':((actual_p>=0)&(actual_p<=1)).all(),'schema_exact':schema.get('channels')==CHANNELS}
replay={'provider_id':policy['provider_id'],'checks':{k:bool(v) for k,v in checks.items()},'passed':bool(all(checks.values())),'model_sha256':sha(model_path),'calibrator_sha256':sha(cal_path),'provider_source_sha256':sha(BUNDLE/'p_print_provider.py')}; writej(REPLAY/'replay_test.json',replay); require(replay['passed'],'Replay failed.'); print(replay)

## 5. Manifest, verdict and review bundle

In [ ]:
env={'python':sys.version,'torch':torch.__version__,'cuda':torch.version.cuda,'gpu':torch.cuda.get_device_name(0),'opencv':cv2.__version__,'numpy':np.__version__,'pandas':pd.__version__,'sklearn':sklearn.__version__,'platform':platform.platform()}; writej(TRAIN/'software_environment.json',env)
fitmanifest={'created_utc':datetime.now(timezone.utc).isoformat(),'source_v82_run':str(SOURCE),'source_identity_sha256':sha(SOURCE/'experiment_identity.json'),'source_schema_sha256':sha(SOURCE/'inputs/channel_schema.json'),'stable_manifest_sha256':sha(SOURCE/'tables/stable_sample_manifest.csv'),'raw_oof_sha256':sha(SOURCE/'tables/stable_raw_oof.npy'),'full_development_refit':True,'parameters_changed':False,'calibration_from_complete_oof':True,'independent_data_used':False,'software':env}; writej(TRAIN/'fitting_manifest.json',fitmanifest)
manifest={'version':'8.2F-full-development-p-print-provider','provider_id':policy['provider_id'],'model':'p_print_model.pt','model_sha256':sha(model_path),'calibrator':'p_print_calibrator.joblib','calibrator_sha256':sha(cal_path),'channel_schema_sha256':sha(BUNDLE/'channel_schema.json'),'provider_policy_sha256':sha(BUNDLE/'provider_policy.json'),'provider_source_sha256':sha(BUNDLE/'p_print_provider.py'),'replay_sha256':sha(REPLAY/'replay_test.json'),'unseen_page_inference':True,'clean_truth_required_at_inference':False,'promotion_authority':False}; manifest['bundle_id']=oh(manifest); writej(BUNDLE/'bundle_manifest.json',manifest)
inv=[]
for p in sorted(BUNDLE.rglob('*')):
 if p.is_file() and p.name!='artifact_inventory.csv': inv.append({'relative_path':str(p.relative_to(BUNDLE)),'bytes':p.stat().st_size,'sha256':sha(p)})
pd.DataFrame(inv).to_csv(BUNDLE/'artifact_inventory.csv',index=False)
checks={'source_contract_exact':True,'population_complete':len(stable)>0,'model_loadable':True,'calibrator_loadable':hasattr(joblib.load(cal_path),'predict'),'schema_exact':schema.get('channels')==CHANNELS,'replay_exact':replay['passed'],'no_independent_data':identity.get('independent_data_used') is False,'runtime_truth_free':policy['clean_truth_required_at_inference'] is False}; first=next((k for k,v in checks.items() if v is not True),None); verdict='PASS_P_PRINT_PROVIDER_READY_FOR_OPERATOR_INTEGRATION' if first is None else 'FAIL_P_PRINT_PROVIDER_FREEZE'; vr={'verdict':verdict,'first_failed_gate':first,'checks':checks,'bundle_id':manifest['bundle_id'],'provider_id':policy['provider_id'],'meaning':'PASS approves this p_print probability provider for operator integration; not production authorisation.'}; writej(OUT/'v8_2f_verdict.json',vr)
lines=['V8.2F FULL-DEVELOPMENT P_PRINT PROVIDER','',f'VERDICT: {verdict}',f'Bundle ID: {manifest["bundle_id"]}',f'Provider ID: {policy["provider_id"]}',f'Training rows: {len(stable)}',f'Pages: {stable.page.nunique()}',f'Source groups: {stable.source_group.nunique()}','Exact V8.2 model parameters: yes','Calibration source: complete source-group OOF raw scores','Exact reload replay: yes','Independent data used: no','Production authority: no']; (OUT/'executive_summary.txt').write_text(chr(10).join(lines))
review=[OUT/'executive_summary.txt',OUT/'v8_2f_verdict.json',BUNDLE/'bundle_manifest.json',BUNDLE/'provider_policy.json',BUNDLE/'channel_schema.json',BUNDLE/'artifact_inventory.csv',REPLAY/'replay_test.json',TRAIN/'fitting_manifest.json',TRAIN/'model_fitting_record.json',TRAIN/'calibrator_fitting_record.json']
for q in review: shutil.copy2(q,UPLOAD/q.name)
pd.DataFrame([{'file':q.name,'bytes':q.stat().st_size,'sha256':sha(q)} for q in review]).to_csv(UPLOAD/'upload_manifest.csv',index=False); (UPLOAD/'README.txt').write_text(chr(10).join(lines)+chr(10))
with zipfile.ZipFile(UPLOAD/'v8_2f_review_bundle.zip','w',zipfile.ZIP_DEFLATED) as z:
 for q in UPLOAD.iterdir():
  if q.is_file() and q.suffix!='.zip': z.write(q,q.name)
print(chr(10).join(lines)); print('Bundle:',BUNDLE); print('Review:',UPLOAD/'v8_2f_review_bundle.zip')
if first: raise RuntimeError(f'Failed at {first}')


## Review order

1. `executive_summary.txt`
2. `v8_2f_verdict.json`
3. `p_print_provider/bundle_manifest.json`
4. `p_print_provider/replay/replay_test.json`
5. `p_print_provider/training_record/fitting_manifest.json`
6. `upload/v8_2f_review_bundle.zip`

The integration artefacts are `p_print_model.pt`, `p_print_calibrator.joblib`, `channel_schema.json`, `provider_policy.json`, and `p_print_provider.py`.